# Enterprise Knowledge Navigator
## Milestone Project 2 - AlgoProfessor Internship 2026
## DAY 1: Setup + Dataset Collection + Ingestion Pipeline
### Stack: sentence-transformers + ChromaDB + BM25 (100% FREE)

**What we build today:**
- Mount Google Drive and create full folder structure
- Download 30+ real Tesla documents (live dataset)
- Build PDF and text loader with metadata extraction
- Intelligent chunking pipeline (500 words, 75 overlap)
- Generate embeddings using sentence-transformers (FREE, no API)
- Store in ChromaDB persistent vector database
- Build BM25 keyword index for hybrid search
- Test both search types and verify they work


## STEP 1: Mount Google Drive

**Why Google Drive?**
ChromaDB stores vector embeddings as files on disk.
Without Drive, all data is lost when Colab session ends.
Mounting Drive means ChromaDB persists between sessions permanently.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

## STEP 2: Create Project Folder Structure

- data/raw/ = original documents, never modify originals
- data/processed/ = extracted clean text and saved embeddings
- chromadb/ = vector database files saved to Drive
- bm25_index/ = keyword search index files
- outputs/ = evaluation results and screenshots for GitHub submission


In [ ]:
import os

BASE = '/content/drive/MyDrive/EnterpriseKnowledgeNavigator'

folders = [
    BASE + '/data/raw',
    BASE + '/data/processed',
    BASE + '/chromadb',
    BASE + '/bm25_index',
    BASE + '/outputs/eval',
    BASE + '/outputs/logs',
    BASE + '/notebooks',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print('Created: ' + folder)

print('Folder structure ready!')

## STEP 3: Install All Libraries

- pymupdf: fastest PDF parser, extracts text page by page with metadata
- sentence-transformers: converts text to embedding vectors locally, FREE
- chromadb: stores and searches embedding vectors, saves to disk
- rank-bm25: keyword search algorithm, BM25 is industry standard
- groq: free LLM API for answer generation on Day 2

**This takes 3-4 minutes. Read the notes above while waiting.**


In [ ]:
!pip install -q pymupdf pdfplumber
!pip install -q sentence-transformers
!pip install -q chromadb
!pip install -q rank-bm25
!pip install -q groq
!pip install -q requests beautifulsoup4 lxml tqdm
print('All libraries installed!')

## STEP 4: Import Libraries and Verify

Always verify imports before expensive operations.
Fix errors now, not after 30 minutes of processing.


In [ ]:
import os, json, pickle, re, time, requests, hashlib
from pathlib import Path
from datetime import datetime
from typing import List, Dict
import fitz
from bs4 import BeautifulSoup
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi
import numpy as np

BASE = '/content/drive/MyDrive/EnterpriseKnowledgeNavigator'
print('All imports OK!')
print('Session: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

## STEP 5: Download Live Tesla Dataset

**Why Tesla?**
Tesla is publicly traded. All financial documents are legally required to be public.
SEC.gov hosts annual reports for free. Tesla has diverse document types.
This is a realistic enterprise knowledge base exactly as the project requires.

**Sources:**
- 25 Wikipedia pages: products, technology, manufacturing, leadership, energy
- 5 structured documents: HR, financials, technology, supercharger, overview
- Total: 30+ real documents ready for ingestion


In [ ]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

WEB_URLS = [
    {'url': 'https://en.wikipedia.org/wiki/Tesla,_Inc.', 'name': 'wiki_tesla_inc', 'category': 'company_overview'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Model_S', 'name': 'wiki_model_s', 'category': 'products'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Model_3', 'name': 'wiki_model_3', 'category': 'products'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Model_X', 'name': 'wiki_model_x', 'category': 'products'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Model_Y', 'name': 'wiki_model_y', 'category': 'products'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Cybertruck', 'name': 'wiki_cybertruck', 'category': 'products'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Autopilot', 'name': 'wiki_autopilot', 'category': 'technology'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Powerwall', 'name': 'wiki_powerwall', 'category': 'energy'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Megapack', 'name': 'wiki_megapack', 'category': 'energy'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Semi', 'name': 'wiki_semi', 'category': 'products'},
    {'url': 'https://en.wikipedia.org/wiki/Gigafactory_1', 'name': 'wiki_gigafactory1', 'category': 'manufacturing'},
    {'url': 'https://en.wikipedia.org/wiki/Gigafactory_Shanghai', 'name': 'wiki_giga_shanghai', 'category': 'manufacturing'},
    {'url': 'https://en.wikipedia.org/wiki/Gigafactory_Berlin', 'name': 'wiki_giga_berlin', 'category': 'manufacturing'},
    {'url': 'https://en.wikipedia.org/wiki/Gigafactory_Texas', 'name': 'wiki_giga_texas', 'category': 'manufacturing'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Supercharger', 'name': 'wiki_supercharger', 'category': 'infrastructure'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_energy', 'name': 'wiki_tesla_energy', 'category': 'energy'},
    {'url': 'https://en.wikipedia.org/wiki/Elon_Musk', 'name': 'wiki_elon_musk', 'category': 'leadership'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Full_Self-Driving', 'name': 'wiki_fsd', 'category': 'technology'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Optimus', 'name': 'wiki_optimus', 'category': 'technology'},
    {'url': 'https://en.wikipedia.org/wiki/Dojo_(Tesla)', 'name': 'wiki_dojo', 'category': 'technology'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Solar_Roof', 'name': 'wiki_solar_roof', 'category': 'energy'},
    {'url': 'https://en.wikipedia.org/wiki/SolarCity', 'name': 'wiki_solarcity', 'category': 'history'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Powerpack', 'name': 'wiki_powerpack', 'category': 'energy'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Roadster_(first_generation)', 'name': 'wiki_roadster1', 'category': 'history'},
    {'url': 'https://en.wikipedia.org/wiki/Tesla_Roadster_(second_generation)', 'name': 'wiki_roadster2', 'category': 'products'},
]

print('Web sources defined: ' + str(len(WEB_URLS)))

In [ ]:
def scrape_wikipedia(url, save_path, name):
    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(response.content, 'html.parser')
        title_el = soup.find('h1', {'id': 'firstHeading'})
        title_text = title_el.get_text() if title_el else name
        content_div = soup.find('div', {'id': 'mw-content-text'})
        if not content_div:
            return False
        paragraphs = content_div.find_all('p')
        text_parts = ['# ' + title_text + '\n\nSource: ' + url + '\n\n']
        for p in paragraphs:
            text = p.get_text().strip()
            if len(text) > 50:
                text = re.sub(r'\[\d+\]', '', text)
                text_parts.append(text + '\n')
        full_text = '\n'.join(text_parts)
        if len(full_text) > 500:
            with open(save_path, 'w', encoding='utf-8') as f:
                f.write(full_text)
            print('  Scraped: ' + title_text + ' (' + str(len(full_text.split())) + ' words)')
            return True
        return False
    except Exception as e:
        print('  ERROR ' + name + ': ' + str(e)[:50])
        return False

print('Scraping Wikipedia pages...')
web_success = 0
for doc in WEB_URLS:
    save_path = BASE + '/data/raw/' + doc['name'] + '.txt'
    if os.path.exists(save_path):
        print('  Already exists: ' + doc['name'])
        web_success += 1
    else:
        if scrape_wikipedia(doc['url'], save_path, doc['name']):
            web_success += 1
    time.sleep(0.5)

print('Scraped: ' + str(web_success) + '/' + str(len(WEB_URLS)) + ' pages')

## Create Additional Structured Tesla Documents

Real enterprise systems have internal docs like HR policies and financial summaries.
These structured text files show you handle diverse document types professionally.


In [ ]:
extra_docs = {}
extra_docs['Tesla_Company_Overview.txt'] = 'Tesla Inc Company Overview

Tesla Inc is an American multinational automotive and clean energy company headquartered in Austin Texas.
Founded in 2003. Tesla mission is to accelerate the worlds transition to sustainable energy.

Products and Services

Automotive vehicles include Model S luxury sedan, Model 3 mid-size sedan best selling EV globally,
Model X luxury SUV, Model Y compact SUV which was worlds best selling car in 2023,
Cybertruck electric pickup, and Semi electric commercial truck.

Energy products include Powerwall home battery, Powerpack commercial battery,
Megapack utility scale battery, Solar panels, and Solar Roof tiles.

Key Financials 2023

Revenue was 96.77 billion dollars. Net income was 14.97 billion dollars.
Vehicles delivered were 1.81 million units. Employees approximately 140000 worldwide.

Manufacturing Locations

Fremont Factory California USA. Gigafactory Nevada for batteries and Model 3.
Gigafactory Shanghai China. Gigafactory Berlin Brandenburg Europe. Gigafactory Texas Austin.
'
extra_docs['Tesla_Technology_Overview.txt'] = 'Tesla Technology Deep Dive

Battery Technology

The 4680 cylindrical battery cell measures 46mm by 80mm dimensions.
It provides 5 times more energy than previous 2170 cells and 6 times more power output.
Tabless design reduces internal resistance significantly.
Structural battery pack integration reduces vehicle weight.

Autopilot and Full Self Driving

Tesla uses vision only approach with 8 cameras providing 360 degree coverage.
Custom FSD chip processes neural network at 72 TOPS tera operations per second.
Fleet learning allows system to learn from millions of real world miles.
Dojo supercomputer accelerates neural network training at scale.

Gigapress Manufacturing

Tesla uses worlds largest die casting machines called Gigapress.
Gigapress casts entire rear underbody as single piece.
This replaces 70 to 100 individual stamped parts.
Reduces manufacturing time and vehicle weight.
'
extra_docs['Tesla_Financial_Data.txt'] = 'Tesla Financial Overview and Investor Information

Revenue History

Tesla revenue in 2019 was 24.6 billion dollars.
Tesla revenue in 2020 was 31.5 billion dollars.
Tesla revenue in 2021 was 53.8 billion dollars.
Tesla revenue in 2022 was 81.5 billion dollars.
Tesla revenue in 2023 was 96.8 billion dollars.

Vehicle Deliveries

Tesla delivered 367500 vehicles in 2019.
Tesla delivered 499550 vehicles in 2020.
Tesla delivered 936172 vehicles in 2021.
Tesla delivered 1313851 vehicles in 2022.
Tesla delivered 1808581 vehicles in 2023.

Stock Information

Tesla ticker symbol is TSLA on NASDAQ exchange.
Tesla was added to SP 500 index in December 2020.

Q4 2023 Results

Tesla delivered 484507 vehicles in Q4 2023 which was a record quarter.
Q4 2023 revenue was 25.2 billion dollars. Free cash flow was 2.1 billion dollars.
'
extra_docs['Tesla_Supercharger_Network.txt'] = 'Tesla Supercharger Network

Network Overview

Tesla Supercharger network has over 50000 Superchargers at 5000 plus stations globally.
Available in over 50 countries. Growing at approximately 1000 new connectors per month.

Charging Speeds

V1 and V2 Superchargers provide 150 kW maximum charging speed.
V3 Superchargers provide 250 kW maximum which is now most common.
V4 Superchargers provide 350 kW maximum and can handle large trucks.

Charging Time with V3

Model 3 can add 170 plus miles in 15 minutes.
Model Y can add 162 plus miles in 15 minutes.

Network Opening to Other Brands

Tesla began opening network to non Tesla vehicles in 2022.
Ford GM Rivian Polestar have adopted Tesla NACS connector standard.
NACS is becoming North American industry standard for EV charging.
'
extra_docs['Tesla_HR_Culture.txt'] = 'Tesla Human Resources and Culture

Core Values

Move Fast: Tesla operates with urgency and decisiveness in all decisions.
Do the Impossible: Tesla tackles engineering challenges others avoid.
Win on Safety: Employee and customer safety is always paramount.
Be a Team Player: Collaboration drives results across all departments.
Think Like an Owner: Every employee is responsible for outcomes.

Hiring Process

Step 1 is initial screening call with recruiter.
Step 2 is technical phone screen interview.
Step 3 is onsite or virtual interview loop with 4 to 6 rounds total.
Step 4 is offer and background check after successful interviews.

Compensation Philosophy

Tesla pays competitive base salaries with significant equity via RSU grants.
Benefits include health dental vision insurance and 401k with company match.
Annual performance reviews determine RSU grants and base salary changes.
'

for filename, content in extra_docs.items():
    with open(BASE + '/data/raw/' + filename, 'w', encoding='utf-8') as f:
        f.write(content)
    print('Created: ' + filename)

all_files = os.listdir(BASE + '/data/raw')
print('Total documents: ' + str(len(all_files)))


## STEP 6: Document Loader with Metadata Extraction

**Why metadata is critical in RAG:**
Metadata enables source citation showing page number and document name.
It allows filtering to search only financial documents for financial questions.
It enables debugging to trace which chunk caused a wrong answer.
Without metadata RAG gives answers with no proof. With metadata it is trustworthy.

**Metadata we extract per chunk:**
source = filename, page_number = exact page, category = document type,
char_count = content size, doc_id = unique hash identifier.


In [ ]:
class TeslaDocumentLoader:
    def __init__(self, data_dir):
        self.data_dir = Path(data_dir)
        self.documents = []
        self.errors = []

    def get_category(self, filename):
        fn = filename.lower()
        if 'impact' in fn or 'sustain' in fn: return 'sustainability'
        if 'model' in fn or 'cyber' in fn or 'semi' in fn or 'roadster' in fn: return 'products'
        if 'giga' in fn or 'manufactur' in fn: return 'manufacturing'
        if 'power' in fn or 'mega' in fn or 'solar' in fn or 'energy' in fn: return 'energy'
        if 'auto' in fn or 'fsd' in fn or 'dojo' in fn or 'techno' in fn or 'optimus' in fn: return 'technology'
        if 'supercharg' in fn: return 'infrastructure'
        if 'elon' in fn or 'leader' in fn: return 'leadership'
        if 'financial' in fn or 'revenue' in fn: return 'financial'
        if 'hr' in fn or 'cultur' in fn: return 'human_resources'
        return 'general'

    def make_id(self, text, source, page):
        content = source + '_' + str(page) + '_' + text[:80]
        return hashlib.md5(content.encode()).hexdigest()[:16]

    def load_pdf(self, pdf_path):
        docs = []
        try:
            pdf = fitz.open(str(pdf_path))
            cat = self.get_category(pdf_path.name)
            for pg in range(len(pdf)):
                text = pdf[pg].get_text().strip()
                if len(text) < 100:
                    continue
                text = re.sub(r'\n{3,}', '\n\n', text)
                docs.append({
                    'text': text,
                    'metadata': {
                        'source': pdf_path.name,
                        'source_type': 'pdf',
                        'page_number': pg + 1,
                        'total_pages': len(pdf),
                        'category': cat,
                        'char_count': len(text),
                        'doc_id': self.make_id(text, pdf_path.name, pg)
                    }
                })
            pdf.close()
        except Exception as e:
            self.errors.append({'file': pdf_path.name, 'error': str(e)})
        return docs

    def load_text(self, txt_path):
        docs = []
        try:
            with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
            if len(text.strip()) < 100:
                return []
            cat = self.get_category(txt_path.name)
            sections = re.split(r'(?=^#{1,3} )', text, flags=re.MULTILINE)
            for i, sec in enumerate(sections):
                sec = sec.strip()
                if len(sec) < 50:
                    continue
                docs.append({
                    'text': sec,
                    'metadata': {
                        'source': txt_path.name,
                        'source_type': 'text',
                        'page_number': i + 1,
                        'total_pages': len(sections),
                        'category': cat,
                        'char_count': len(sec),
                        'doc_id': self.make_id(sec, txt_path.name, i)
                    }
                })
        except Exception as e:
            self.errors.append({'file': txt_path.name, 'error': str(e)})
        return docs

    def load_all(self):
        pdf_files = list(self.data_dir.glob('*.pdf'))
        txt_files = list(self.data_dir.glob('*.txt'))
        print('Found: ' + str(len(pdf_files)) + ' PDFs, ' + str(len(txt_files)) + ' text files')
        for f in pdf_files:
            docs = self.load_pdf(f)
            self.documents.extend(docs)
            if docs:
                print('  PDF: ' + f.name + ' -> ' + str(len(docs)) + ' pages')
        for f in txt_files:
            docs = self.load_text(f)
            self.documents.extend(docs)
            if docs:
                print('  TXT: ' + f.name + ' -> ' + str(len(docs)) + ' sections')
        return self.documents


loader = TeslaDocumentLoader(BASE + '/data/raw')
raw_documents = loader.load_all()
print('Total sections loaded: ' + str(len(raw_documents)))

## STEP 7: Intelligent Chunking

**Why chunking is the most critical RAG decision:**
Too large means diluted embeddings and bad search results.
Too small means no context and LLM cannot form a good answer.
Sweet spot is 400 to 600 words with 75 word overlap.

**Overlap prevents split answers:**
Consecutive chunks share 75 words ensuring no answer is cut at a boundary.
Think of it as a sliding window that moves forward but keeps some previous text.


In [ ]:
class RecursiveChunker:
    def __init__(self, chunk_size=500, overlap=75):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def chunk_document(self, text, metadata):
        paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 30]
        if not paragraphs:
            return []
        chunks = []
        current_words = []
        chunk_index = 0
        for para in paragraphs:
            para_words = para.split()
            if len(current_words) + len(para_words) > self.chunk_size and current_words:
                chunk_text = ' '.join(current_words)
                meta = dict(metadata)
                meta['chunk_index'] = chunk_index
                meta['chunk_word_count'] = len(current_words)
                meta['chunk_id'] = metadata['doc_id'] + '_c' + str(chunk_index)
                chunks.append({'text': chunk_text, 'metadata': meta})
                chunk_index += 1
                if len(current_words) > self.overlap:
                    current_words = current_words[-self.overlap:]
                else:
                    current_words = []
            current_words.extend(para_words)
        if current_words:
            chunk_text = ' '.join(current_words)
            meta = dict(metadata)
            meta['chunk_index'] = chunk_index
            meta['chunk_word_count'] = len(current_words)
            meta['chunk_id'] = metadata['doc_id'] + '_c' + str(chunk_index)
            chunks.append({'text': chunk_text, 'metadata': meta})
        return chunks

    def chunk_all(self, documents):
        all_chunks = []
        for doc in tqdm(documents, desc='Chunking'):
            chunks = self.chunk_document(doc['text'], doc['metadata'])
            all_chunks.extend(chunks)
        return all_chunks


chunker = RecursiveChunker(chunk_size=500, overlap=75)
all_chunks = chunker.chunk_all(raw_documents)
print('Total chunks: ' + str(len(all_chunks)))
avg = sum(c['metadata']['chunk_word_count'] for c in all_chunks) // max(len(all_chunks), 1)
print('Average chunk size: ' + str(avg) + ' words')

ex = all_chunks[3]
print('\nExample chunk:')
print('Source: ' + ex['metadata']['source'] + ' | Page: ' + str(ex['metadata']['page_number']))
print('Category: ' + ex['metadata']['category'])
print('Preview: ' + ex['text'][:300])

## STEP 8: Generate Embeddings (FREE with sentence-transformers)

**What are embeddings?**
Converts text into a list of numbers that captures meaning.
Similar meaning = similar numbers = found by vector search.

Example:
Tesla makes electric cars -> [0.23, -0.45, 0.81, ...] 384 numbers
Tesla produces EVs        -> [0.22, -0.43, 0.79, ...] very similar
I enjoy eating pizza      -> [-0.67, 0.23, ...]        very different

**Why sentence-transformers not OpenAI?**
Completely FREE, runs locally on GPU, no API cost, production quality.
all-MiniLM-L6-v2 is widely used in production systems at top companies.

**Go to Runtime > Change runtime type > T4 GPU (free) for 5x speed**


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device: ' + device)
if device == 'cuda':
    print('GPU: ' + torch.cuda.get_device_name(0))

print('Loading embedding model...')
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print('Model loaded. Dimensions: ' + str(embedding_model.get_sentence_embedding_dimension()))

texts = [c['text'] for c in all_chunks]
print('Generating embeddings for ' + str(len(texts)) + ' chunks...')
print('Estimated: 5-10 min GPU, 15-20 min CPU')

start = time.time()
embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)
elapsed = time.time() - start
print('Done in ' + str(round(elapsed, 1)) + 's | Shape: ' + str(embeddings.shape))

from numpy import dot
from numpy.linalg import norm
t1 = embedding_model.encode('Tesla makes electric vehicles', normalize_embeddings=True)
t2 = embedding_model.encode('Tesla produces electric cars', normalize_embeddings=True)
t3 = embedding_model.encode('I enjoy eating pizza', normalize_embeddings=True)
print('Sanity check:')
print('  Tesla EVs vs Tesla cars: ' + str(round(float(dot(t1,t2)), 3)) + ' (should be ~0.9)')
print('  Tesla EVs vs Pizza:      ' + str(round(float(dot(t1,t3)), 3)) + ' (should be ~0.1)')

## STEP 9: Store in ChromaDB (Persistent Vector Database)

**What is ChromaDB?**
A vector database that stores embeddings and lets you search them.
Query Tesla revenue -> embed query -> find nearest vectors -> return relevant chunks.

PersistentClient saves all data to Google Drive so nothing is lost between sessions.


In [ ]:
chroma_path = BASE + '/chromadb'
chroma_client = chromadb.PersistentClient(path=chroma_path)

try:
    chroma_client.delete_collection('tesla_knowledge_base')
    print('Deleted old collection')
except:
    pass

collection = chroma_client.create_collection(
    name='tesla_knowledge_base',
    metadata={'hnsw:space': 'cosine'}
)
print('Collection created!')

BATCH = 500
for i in tqdm(range(0, len(all_chunks), BATCH), desc='Inserting'):
    batch = all_chunks[i:i+BATCH]
    emb_batch = embeddings[i:i+BATCH]
    collection.add(
        ids=[c['metadata']['chunk_id'] for c in batch],
        embeddings=emb_batch.tolist(),
        documents=[c['text'] for c in batch],
        metadatas=[c['metadata'] for c in batch]
    )

print('ChromaDB ready! Vectors stored: ' + str(collection.count()))

def test_vector(query, n=3):
    qe = embedding_model.encode(query, normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=[qe], n_results=n,
                           include=['documents', 'metadatas', 'distances'])
    print('Query: ' + query)
    for doc, meta, dist in zip(res['documents'][0], res['metadatas'][0], res['distances'][0]):
        sim = round(1 - dist, 3)
        print('  [' + meta['category'] + '] ' + meta['source'] + ' p' + str(meta['page_number']) + ' sim=' + str(sim))
        print('  ' + doc[:150] + '...')
        print()

print('\n--- TEST: Vector Search ---')
test_vector('What is Tesla annual revenue?')
test_vector('How does Tesla Autopilot work?')

## STEP 10: Build BM25 Keyword Index

**Why BM25 in addition to vector search?**
Vector search finds semantic matches but can miss exact keywords.
Query Q4 2023 revenue exactly needs keyword matching for specific numbers and dates.
BM25 finds exact keyword matches while vector finds semantic matches.
Hybrid Search combines both which is what top production RAG systems use.


In [ ]:
def tokenize_bm25(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
    return [t for t in text.split() if len(t) > 1]

print('Building BM25 index...')
tokenized = [tokenize_bm25(c['text']) for c in tqdm(all_chunks, desc='Tokenizing')]
bm25 = BM25Okapi(tokenized)
print('BM25 index built!')

bm25_path = BASE + '/bm25_index/bm25_index.pkl'
with open(bm25_path, 'wb') as f:
    pickle.dump({'index': bm25, 'chunks': all_chunks, 'tokenized': tokenized}, f)
size = os.path.getsize(bm25_path) / 1e6
print('Saved to Drive: ' + str(round(size, 1)) + ' MB')

def test_bm25(query, n=3):
    tokens = tokenize_bm25(query)
    scores = bm25.get_scores(tokens)
    top = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n]
    print('BM25 Query: ' + query)
    for idx in top:
        c = all_chunks[idx]
        print('  [' + c['metadata']['category'] + '] ' + c['metadata']['source'] + ' score=' + str(round(scores[idx], 2)))
        print('  ' + c['text'][:150] + '...')
        print()

print('\n--- TEST: BM25 Search ---')
test_bm25('Q4 2023 revenue billion')
test_bm25('4680 battery cell specifications')

## STEP 11: Save Everything to Drive


In [ ]:
with open(BASE + '/data/processed/all_chunks.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)
np.save(BASE + '/data/processed/embeddings.npy', embeddings)
print('Chunks and embeddings saved to Drive')

summary = {
    'date': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'total_chunks': len(all_chunks),
    'embedding_model': 'all-MiniLM-L6-v2',
    'embedding_dims': int(embeddings.shape[1]),
    'chroma_vectors': collection.count(),
    'bm25_indexed': len(tokenized),
    'device': device,
    'status': 'DAY 1 COMPLETE'
}
with open(BASE + '/outputs/day1_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 50)
print('DAY 1 COMPLETE!')
print('=' * 50)
print(json.dumps(summary, indent=2))
print('Next: File > Download > Download .ipynb and commit to GitHub')
print('Commit: [Day-15] Enterprise Knowledge Navigator - Day1 Ingestion Pipeline')

## DAY 1 COMPLETE!

### What you built:
- 30+ real Tesla documents collected from Wikipedia and structured sources
- PDF and text loader with full metadata extraction per page
- Intelligent chunker with 500 word chunks and 75 word overlap
- sentence-transformers embeddings FREE and GPU accelerated
- ChromaDB persistent vector store saved to Google Drive
- BM25 keyword index saved to Google Drive
- Both search types tested and verified working

### Key concepts you learned:
1. Why metadata matters: source citations, filtering, debugging
2. Chunking strategy: why 500 words and 75 overlap not random numbers
3. Embeddings: text to numbers, similar text = similar numbers
4. Vector search vs BM25 keyword search and their different strengths
5. Why hybrid search beats either method alone
6. Persistent storage: saving ChromaDB to Google Drive

### Before closing Colab:
1. File > Download > Download .ipynb
2. ChromaDB and BM25 data already saved in Drive automatically
3. Push notebook to GitHub
4. Commit message: [Day-15] Enterprise Knowledge Navigator - Day1 Ingestion Pipeline

### Day 2 preview:
- HyDE retrieval: generate hypothetical answer then search for it
- Multi-query: ask 3 variations of the question and merge results
- Hybrid Search: combine BM25 and Vector with RRF score fusion
- Neo4j Graph RAG: extract entities and build knowledge graph
- Full RAG pipeline with Groq LLM free: query to answer with page citations
